In [9]:
#!pip install langgraph langchain_openai sentence_transformers faiss-cpu

In [2]:
import operator
from typing import Annotated, List, TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
# --- 1. State Definition ---
class AgentState(TypedDict):
    query: str
    market_data: str
    candidates: List[str]
    refined_matches: List[dict]
    iteration: int

In [4]:
# --- 2. Two-Stage Matching Logic ---
bi_encoder = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [5]:
def retrieval_stage(state: AgentState):
    # Simulated Bi-Encoder Retrieval (Top 10 from 1000s)
    print("--- Stage 1: Bi-Encoder Retrieval ---")
    return {"candidates": ["Candidate A (Python)", "Candidate B (C++)", "Candidate C (ML)"]}

In [6]:
def ranking_stage(state: AgentState):
    # Stage 2: Precision Cross-Encoder Ranking
    print("--- Stage 2: Cross-Encoder Re-ranking ---")
    job_desc = state['query']
    candidates = state['candidates']

    # Predict scores for (Query, Candidate) pairs
    scores = cross_encoder.predict([(job_desc, c) for c in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return {"refined_matches": [{"name": r[0], "score": float(r[1])} for r in ranked]}

In [7]:
# --- 3. LangGraph Workflow ---
workflow = StateGraph(AgentState)

workflow.add_node("market_research", lambda x: {"market_data": "High demand for LLM Engineers"})
workflow.add_node("retrieve", retrieval_stage)
workflow.add_node("rank", ranking_stage)

workflow.set_entry_point("market_research")
workflow.add_edge("market_research", "retrieve")
workflow.add_edge("retrieve", "rank")
workflow.add_edge("rank", END)

app = workflow.compile()

In [8]:
# Run the system
inputs = {"query": "Senior Machine Learning Engineer with LangChain experience", "iteration": 0}
for output in app.stream(inputs):
    print(output)

{'market_research': {'market_data': 'High demand for LLM Engineers'}}
--- Stage 1: Bi-Encoder Retrieval ---
{'retrieve': {'candidates': ['Candidate A (Python)', 'Candidate B (C++)', 'Candidate C (ML)']}}
--- Stage 2: Cross-Encoder Re-ranking ---
{'rank': {'refined_matches': [{'name': 'Candidate C (ML)', 'score': -10.810101509094238}, {'name': 'Candidate B (C++)', 'score': -11.098665237426758}, {'name': 'Candidate A (Python)', 'score': -11.269366264343262}]}}
